# LegalQA Main Stage 3/3 — generate, submission và diagnostics

Notebook này chỉ load adapter đã chọn và cache retrieval public từ Stage 2, sinh 1000 câu trả lời, kiểm tra schema, tạo `submission.zip` và `diagnostics.zip`.

Trước khi chạy, Add Input output của `legalqa_main_02_select_retrieve.ipynb` hoặc dataset được tạo từ output đó. Generation tự ghi journal checkpoint sau từng câu và có thể tiếp tục khi chạy lại cell trong cùng session.

## 1. Cấu hình và tìm Stage 2 input

In [ ]:
from pathlib import Path
import hashlib, json, shutil, subprocess, sys
from zipfile import ZIP_DEFLATED, ZipFile

if not Path('/kaggle').exists():
    raise RuntimeError('Notebook này chỉ chạy trên Kaggle.')

WORK_BASE = Path('/kaggle/working')
INPUT_BASE = Path('/kaggle/input')
REPO_URL = 'https://github.com/lighth-gh/uit-dsc-2026-task2-legalqa.git'
REPO_REF = 'main'
CODE = WORK_BASE / 'uit-dsc-2026-task2-legalqa'
VERSION3_ROOT = Path('/kaggle/input/datasets/lighth/ver3-smoke-output/legalqa_smoke_full_v1')
SAVED_MODELS = VERSION3_ROOT / 'models'

STAGE2_INPUT_ROOT = None  # Nếu có nhiều bản Stage 2, đặt Path('/kaggle/input/.../legalqa_main_stage2_select_retrieve_v8').
RUN_ROOT = WORK_BASE / 'legalqa_main_stage3_generate_submit_v8'
MODELS = RUN_ROOT / 'models'
CFG = RUN_ROOT / 'config.json'
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def resolve_input_root(configured, marker):
    if configured is not None:
        root = Path(configured)
        if not (root / marker).is_file():
            raise FileNotFoundError(f'{root / marker} không tồn tại')
        return root
    matches = sorted(INPUT_BASE.rglob(marker))
    if len(matches) != 1:
        raise RuntimeError(f'Cần đúng 1 input chứa {marker}; tìm thấy {len(matches)}: {matches}')
    return matches[0].parent

STAGE2_ROOT = resolve_input_root(STAGE2_INPUT_ROOT, 'stage2_manifest.json')
print('Stage 2 input:', STAGE2_ROOT)
print('Stage 3 output:', RUN_ROOT)

## 2. Clone code và xác thực artifact

In [ ]:
if CODE.exists():
    if not (CODE / '.git').is_dir():
        raise RuntimeError(f'{CODE} tồn tại nhưng không phải Git repo. Hãy Restart Session.')
    subprocess.run(['git', '-C', str(CODE), 'pull', '--ff-only', 'origin', REPO_REF], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(CODE)], check=True)

for path in [SAVED_MODELS / 'models.lock.json', SAVED_MODELS / 'generator' / 'config.json']:
    if not path.is_file():
        raise FileNotFoundError(path)
MODELS.mkdir(parents=True, exist_ok=True)
for role in ['embedding', 'reranker', 'generator']:
    source, link = SAVED_MODELS / role, MODELS / role
    if not (source / 'config.json').is_file():
        raise FileNotFoundError(source / 'config.json')
    if not link.exists():
        link.symlink_to(source, target_is_directory=True)
shutil.copy2(SAVED_MODELS / 'models.lock.json', MODELS / 'models.lock.json')

shared_cfg = json.loads((CODE / 'config.json').read_text(encoding='utf-8'))
if shared_cfg['evaluation'].get('primary_metric') != 'meteor' or shared_cfg['evaluation'].get('target_meteor') != 0.65:
    raise RuntimeError('Cấu hình phải ưu tiên METEOR và target 0.65.')
CFG.write_text(json.dumps(shared_cfg, ensure_ascii=False, indent=2), encoding='utf-8')

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def run(*args):
    command = [sys.executable, '-m', 'legalqa', '--config', str(CFG), '--models', str(MODELS), *map(str, args)]
    print('Running:', ' '.join(command), flush=True)
    subprocess.run(command, cwd=CODE, check=True)

commit = subprocess.check_output(['git', '-C', str(CODE), 'rev-parse', '--short', 'HEAD'], text=True).strip()
stage2_manifest = json.loads((STAGE2_ROOT / 'stage2_manifest.json').read_text(encoding='utf-8'))
if stage2_manifest.get('quality_version') != 'v8':
    raise RuntimeError(f'Stage 2 quality version sai: {stage2_manifest}')
if stage2_manifest.get('code_commit') != commit:
    raise RuntimeError(f'Commit lệch: Stage 2={stage2_manifest.get("code_commit")}, hiện tại={commit}')
if stage2_manifest.get('config_sha256') != file_sha256(CFG):
    raise RuntimeError('config.json khác Stage 2.')

SELECTED_ADAPTER = STAGE2_ROOT / stage2_manifest['selected_adapter']
PUBLIC_QUESTIONS = STAGE2_ROOT / stage2_manifest['public_questions']
PUBLIC_CACHE = STAGE2_ROOT / stage2_manifest['public_retrieval']
for path in [SELECTED_ADAPTER / 'adapter_config.json', SELECTED_ADAPTER / 'adapter_model.safetensors', PUBLIC_QUESTIONS, PUBLIC_CACHE, STAGE2_ROOT / 'selection.json']:
    if not path.is_file():
        raise FileNotFoundError(path)
if file_sha256(PUBLIC_CACHE) != stage2_manifest['public_retrieval_sha256']:
    raise RuntimeError('public.retrieval.json sai hash so với Stage 2 manifest.')
print('Commit:', commit)
print('Selected adapter:', SELECTED_ADAPTER)
print('Evaluation objective:', shared_cfg['evaluation'])

## 3. Cài môi trường và kiểm định

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=CODE, check=True)
(RUN_ROOT / 'environment.freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True), encoding='utf-8')

## 4. Generate 1000 câu public và đóng submission

In [ ]:
DESTINATION = RUN_ROOT / 'submissions' / 'public'
DESTINATION.mkdir(parents=True, exist_ok=True)
PREDICTIONS = DESTINATION / 'submission.json'
SUBMISSION_ZIP = DESTINATION / 'submission.zip'

run('generate', '--questions', PUBLIC_QUESTIONS, '--retrieval', PUBLIC_CACHE,
    '--adapter', SELECTED_ADAPTER, '--output', PREDICTIONS)
run('package', '--predictions', PREDICTIONS, '--questions', PUBLIC_QUESTIONS,
    '--output', SUBMISSION_ZIP, '--filename', 'submission.json')
if not SUBMISSION_ZIP.is_file():
    raise RuntimeError('Không tạo được submission.zip')
predictions = json.loads(PREDICTIONS.read_text(encoding='utf-8'))
questions = json.loads(PUBLIC_QUESTIONS.read_text(encoding='utf-8'))
if len(predictions) != len(questions) or set(predictions) != set(questions):
    raise RuntimeError('Submission không khớp tập ID public.')
print('Submission records:', len(predictions))
print('Submission ZIP:', SUBMISSION_ZIP)

## 5. Tạo diagnostics.zip

In [ ]:
selection = json.loads((STAGE2_ROOT / 'selection.json').read_text(encoding='utf-8'))
DIAGNOSTICS_MANIFEST = RUN_ROOT / 'diagnostics_manifest.json'
DIAGNOSTICS_ZIP = RUN_ROOT / 'legalqa_main_quality_v8_public_diagnostics.zip'
diagnostics_manifest = {
    'stage': 3,
    'quality_version': 'v8',
    'phase': 'public',
    'code_commit': commit,
    'config_sha256': file_sha256(CFG),
    'stage2_manifest_sha256': file_sha256(STAGE2_ROOT / 'stage2_manifest.json'),
    'selected_label': stage2_manifest['selected_label'],
    'selected_meteor': stage2_manifest['selected_meteor'],
    'adapter_sha256': file_sha256(SELECTED_ADAPTER / 'adapter_model.safetensors'),
    'public_retrieval_sha256': file_sha256(PUBLIC_CACHE),
    'submission_sha256': file_sha256(PREDICTIONS),
    'submission_records': len(predictions),
    'weights_included': False,
    'retrieval_payload_included': False,
}
DIAGNOSTICS_MANIFEST.write_text(json.dumps(diagnostics_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

diagnostic_files = [
    DIAGNOSTICS_MANIFEST, CFG, RUN_ROOT / 'environment.freeze.txt',
    STAGE2_ROOT / 'stage2_manifest.json', STAGE2_ROOT / 'selection.json',
    SELECTED_ADAPTER / 'adapter_config.json', SELECTED_ADAPTER / 'trainer_state.json',
    PREDICTIONS, PREDICTIONS.with_suffix('.audit.json'), PREDICTIONS.with_suffix('.manifest.json'),
]
with ZipFile(DIAGNOSTICS_ZIP, 'w', compression=ZIP_DEFLATED) as archive:
    for path in diagnostic_files:
        if path.is_file():
            prefix = 'stage2/' if str(path).startswith(str(STAGE2_ROOT)) else ''
            archive.write(path, arcname=prefix + path.name)
print(json.dumps(diagnostics_manifest, ensure_ascii=False, indent=2))
print('Diagnostics ZIP:', DIAGNOSTICS_ZIP)
print('SUCCESS Stage 3.')